![image.png](https://i.imgur.com/a3uAqnb.png)

# Named Entity Recognition (NER) with Transformers

## 🎯 Objective
Fine-tune a pre-trained Transformer model (DistilBERT) on the **CoNLL-2003** dataset to identify named entities in text:
- **PER** - Person
- **ORG** - Organization
- **LOC** - Location
- **MISC** - Miscellaneous

## 📚 What You'll Learn
- Token-level classification (different from sequence classification!)
- Aligning labels with sub-word tokens
- Working with BIO tagging scheme (B-, I-, O-)
- Using `seqeval` for NER-specific evaluation

---

## Step 1: Install required libraries

In [1]:
!pip install transformers datasets evaluate accelerate seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00


## Step 2: Import libraries

In [2]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)
import evaluate

print("GPU available:", torch.cuda.is_available())

GPU available: False


## Step 3: Load the CoNLL-2003 dataset

In [3]:
dataset = load_dataset("lhoestq/conll2003")
print(dataset)

# Define labels manually (this version doesn't store them as ClassLabel)
label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
print("\nNER labels:", label_list)

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# View an example
example = dataset["train"][0]
print("\nSample sentence:")
for token, tag in zip(example["tokens"], example["ner_tags"]):
    print(f"  {token:15} → {label_list[tag]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

NER labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

Sample sentence:
  EU              → B-ORG
  rejects         → O
  German          → B-MISC
  call            → O
  to              → O
  boycott         → O
  British         → B-MISC
  lamb            → O
  .               → O


## Step 4: Load tokenizer and align labels with sub-words

**Important:** When a tokenizer splits a word like "Washington" into ["Wash", "##ington"], we need to align labels with each sub-token. We assign `-100` to ignored tokens (special tokens or sub-words).

In [4]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    ...
    ...
    ...

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Use a subset for speed 5000 for train, 1000 for test
train_dataset =
eval_dataset =

train_tokenized =
eval_tokenized =

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print("Tokenization done ✅")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenization done ✅


## Step 5: Load the pre-trained model

In [5]:
model = ....

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Step 6: Define NER evaluation metrics (using seqeval)

In [6]:
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored tokens (-100) and convert IDs back to label strings
    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

## Step 7: Train the model

In [ ]:
training_args = ...

trainer = ...

trainer.train()

## Step 8: Evaluate

In [ ]:
results = trainer.evaluate()
print("Final NER Results:")
for key, value in results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

Epoch,Training Loss,Validation Loss


## Step 9: Use the model for inference

In [ ]:
ner_pipeline = pipeline(
    "ner",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"  # Merges B-PER + I-PER into one entity
)

test_sentences = [
    "Elon Musk founded SpaceX in Hawthorne, California.",
    "Apple announced new products at their event in Cupertino last week.",
    "Cristiano Ronaldo plays for Al Nassr in Saudi Arabia.",
    "The United Nations meeting was held in New York City."
]

for sentence in test_sentences:
    print(f"\nSentence: {sentence}")
    entities = ner_pipeline(sentence)
    for entity in entities:
        print(f"  → '{entity['word']}' is a {entity['entity_group']} (confidence: {entity['score']:.3f})")

## Step 10: Visualize entities (Optional - Beautiful Output)

In [ ]:
from IPython.display import HTML, display

colors = {
    "PER": "#FFB6C1",
    "ORG": "#87CEEB",
    "LOC": "#90EE90",
    "MISC": "#FFD700"
}

def visualize_entities(text, entities):
    html = text
    # Sort by start position descending so insertions don't break indices
    for ent in sorted(entities, key=lambda x: x["start"], reverse=True):
        color = colors.get(ent["entity_group"], "#DDDDDD")
        word = text[ent["start"]:ent["end"]]
        span = f'<span style="background-color:{color};padding:2px 6px;border-radius:4px;margin:0 2px">{word} <b>[{ent["entity_group"]}]</b></span>'
        html = html[:ent["start"]] + span + html[ent["end"]:]
    return html

for sentence in test_sentences:
    entities = ner_pipeline(sentence)
    display(HTML(f"<p style='font-size:16px'>{visualize_entities(sentence, entities)}</p>"))

## 📝 For further Exploration

1. **Try with a different model** like `bert-base-cased` (case matters in NER!) and compare.
2. **Train on the full dataset** and measure improvement in F1 score.
3. **Build a custom dataset**: write 10 sentences with manually labeled entities and test the model.
4. **Compare BIO tagging**: print examples of B-PER followed by I-PER, and explain the difference.
5. **Error Analysis**: find 5 sentences where the model fails and discuss why.

## Concepts to Discuss
- Why do we use `-100` for sub-word labels?
- What is the BIO scheme and why do we need it?
- How is token classification different from sequence classification?